# CausalBench Data Generation

This notebook guides you through generating the **CausalBench dataset** - agent trajectories with real API execution.

## What You'll Generate

- **Synthetic trajectories** with real API calls (GitHub, Slack, Stripe, etc.)
- **Attack scenarios** with varying sophistication levels
- **Causal graphs** automatically constructed from execution traces
- **Validated datasets** ready for training detectors

## Prerequisites

- API credentials for services you want to use (optional - falls back to mock data)
- Docker (recommended) OR Python 3.8+ with pip

## Option 1: Quick Start with Pre-Generated Data

Use the sample data already included in this repository:

In [ ]:
%%bash
# Check if sample data exists
cd ..
if [ -d "data/sample" ]; then
    echo "✓ Sample data found"
    echo "  - Attack graphs: $(ls data/sample/attack/*.json 2>/dev/null | wc -l)"
    echo "  - Benign graphs: $(ls data/sample/benign/*.json 2>/dev/null | wc -l)"
    echo ""
    echo "You can skip data generation and proceed to notebook 01."
else
    echo "✗ Sample data not found"
    echo "  → Continue with Option 2 or 3 below to generate data"
fi

## Option 2: Generate with Docker (Recommended)

### Step 1: Check Docker Installation

In [ ]:
%%bash
# Check Docker
if command -v docker &> /dev/null; then
    echo "✓ Docker installed: $(docker --version)"
    
    if command -v docker-compose &> /dev/null; then
        echo "✓ Docker Compose installed: $(docker-compose --version)"
    else
        echo "✗ Docker Compose not found"
        echo "  Install: https://docs.docker.com/compose/install/"
    fi
else
    echo "✗ Docker not installed"
    echo "  Install: https://docs.docker.com/get-docker/"
    echo "  OR use Option 3 (local Python) instead"
fi

### Step 2: Configure Environment (Optional)

If you have API keys, configure them. Otherwise skip this step (will use mock data).

In [ ]:
%%bash
cd ../causalbench-generator

# Copy example env file
if [ ! -f ".env" ]; then
    cp .env.example .env
    echo "✓ Created .env file"
    echo ""
    echo "Edit .env with your API keys (optional):"
    echo "  - GITHUB_TOKEN=ghp_your_token"
    echo "  - SLACK_TOKEN=xoxb_your_token"
    echo "  - STRIPE_API_KEY=sk_test_your_key (MUST be test mode)"
    echo ""
    echo "Or leave empty to use mock data (faster, but less realistic)"
else
    echo "✓ .env file already exists"
fi

### Step 3: Generate 100 Trajectories with Docker

In [ ]:
%%bash
cd ../causalbench-generator

echo "Starting data generation with Docker..."
echo "This will generate 100 trajectories (50 attack, 50 benign)"
echo ""

# Update docker-compose command to generate 100 instead of default 1000
docker-compose run --rm generator \
    --num-trajectories 100 \
    --attack-rate 0.5 \
    --output-dir /app/output

echo ""
echo "✓ Generation complete!"
echo "  Output: causalbench-generator/output/"

## Option 3: Generate with Local Python

### Step 1: Install Dependencies

In [ ]:
%%bash
cd ../causalbench-generator

echo "Installing CausalBench generator dependencies..."
pip install -q -r requirements.txt

echo "✓ Dependencies installed"

### Step 2: Generate 100 Trajectories

In [ ]:
%%bash
cd ../causalbench-generator

echo "Generating 100 trajectories..."
echo "Using simulated writes (faster, no real API calls)"
echo ""

python generate.py \
    --num-trajectories 100 \
    --attack-rate 0.5 \
    --output-dir output/quickstart \
    --simulate-writes

echo ""
echo "✓ Generation complete!"
echo "  Output: causalbench-generator/output/quickstart/"

## Verify Generated Data

Check that data was generated successfully:

In [ ]:
import json
from pathlib import Path

# Check for output directory
output_dirs = [
    Path("../causalbench-generator/output"),
    Path("../causalbench-generator/output/quickstart")
]

for output_dir in output_dirs:
    if output_dir.exists():
        print(f"✓ Found output directory: {output_dir}")
        
        # Check for trajectories
        traj_dir = output_dir / "trajectories"
        if traj_dir.exists():
            traj_files = list(traj_dir.glob("*.json"))
            print(f"  - Trajectories: {len(traj_files)}")
        
        # Check for graphs
        graph_dir = output_dir / "graphs"
        if graph_dir.exists():
            graph_files = list(graph_dir.glob("*.json"))
            print(f"  - Graphs: {len(graph_files)}")
        
        # Load summary
        summary_file = output_dir / "generation_summary.json"
        if summary_file.exists():
            with open(summary_file) as f:
                summary = json.load(f)
            print(f"  - Attack rate: {summary.get('attack_rate', 0):.1%}")
            print(f"  - Generated at: {summary.get('generated_at', 'unknown')}")
        
        print()
        break
else:
    print("✗ No output found. Run Option 2 or 3 above to generate data.")

## Inspect Sample Trajectory

Look at a generated attack trajectory:

In [ ]:
import json
from pathlib import Path

# Find first trajectory file
output_dirs = [
    Path("../causalbench-generator/output"),
    Path("../causalbench-generator/output/quickstart")
]

for output_dir in output_dirs:
    traj_dir = output_dir / "trajectories"
    if traj_dir.exists():
        traj_files = list(traj_dir.glob("*.json"))
        if traj_files:
            # Load first trajectory
            with open(traj_files[0]) as f:
                traj = json.load(f)
            
            print("Sample Trajectory:")
            print("="*70)
            print(f"Session ID: {traj.get('session_id', 'N/A')}")
            print(f"Is Attack: {traj.get('is_attack', False)}")
            print(f"Attack Type: {traj.get('attack_type', 'N/A')}")
            print(f"Sophistication: {traj.get('sophistication', 'N/A')}")
            print(f"Task: {traj.get('task_description', 'N/A')[:80]}...")
            print(f"\nEvents: {len(traj.get('trajectory', []))}")
            print()
            
            # Show first 3 events
            for i, event in enumerate(traj.get('trajectory', [])[:3]):
                print(f"Event {i+1}:")
                print(f"  Type: {event.get('event_type')}")
                print(f"  Service: {event.get('service')}")
                print(f"  Trust Level: {event.get('trust_level', 'N/A')}")
                if event.get('is_injection_source'):
                    print(f"  ⚠️  INJECTION SOURCE")
                print()
            
            break
else:
    print("No trajectories found. Generate data first.")

## Convert to CausalTrace Format

Convert CausalBench trajectories to CausalTrace graph format for analysis:

In [ ]:
%%bash
cd ../causalbench-generator

echo "Converting trajectories to CausalTrace format..."

python -c "
import json
from pathlib import Path

# This is a placeholder - actual conversion is done by CausalTrace extractors
print('Conversion script placeholder')
print('Use CausalTrace extractors to convert CausalBench data')
"

## Validate Generated Dataset

In [ ]:
%%bash
cd ../causalbench-generator

# Find output directory
if [ -d "output/quickstart" ]; then
    OUTPUT_DIR="output/quickstart"
elif [ -d "output" ]; then
    OUTPUT_DIR="output"
else
    echo "✗ No output directory found"
    exit 1
fi

echo "Validating dataset in $OUTPUT_DIR..."
echo ""

python generate.py --validate --input-dir "$OUTPUT_DIR"

echo ""
echo "✓ Validation complete!"
echo "  See: $OUTPUT_DIR/validation_report.json"

## Copy to Main Data Directory

Copy generated data to the main `data/` directory for use in other notebooks:

In [ ]:
%%bash
cd ..

# Create data/causalbench directory
mkdir -p data/causalbench/attack
mkdir -p data/causalbench/benign

# Copy graphs (if they exist)
if [ -d "causalbench-generator/output/graphs" ]; then
    cp causalbench-generator/output/graphs/*.json data/causalbench/ 2>/dev/null || true
    echo "✓ Copied graphs to data/causalbench/"
elif [ -d "causalbench-generator/output/quickstart/graphs" ]; then
    cp causalbench-generator/output/quickstart/graphs/*.json data/causalbench/ 2>/dev/null || true
    echo "✓ Copied graphs to data/causalbench/"
else
    echo "✗ No graphs found to copy"
fi

# Count files
TOTAL=$(ls data/causalbench/*.json 2>/dev/null | wc -l)
echo "  Total graphs: $TOTAL"

## Generate Larger Datasets (Optional)

For reproducing paper results, generate 10,000+ trajectories:

### Docker (Recommended for large datasets)

```bash
cd causalbench-generator

# Generate 10,000 trajectories
docker-compose run --rm generator \
    --num-trajectories 10000 \
    --attack-rate 0.5 \
    --output-dir /app/output/large

# Or use batch profile for 50,000
docker-compose --profile batch up batch-generator
```

### Local Python (for testing)

```bash
cd causalbench-generator

# Generate with simulated writes (faster)
python generate.py \
    --num-trajectories 10000 \
    --attack-rate 0.5 \
    --output-dir output/large \
    --simulate-writes
```

**Note:** Large datasets take hours to generate with real API calls. Use `--simulate-writes` for faster testing.

## Next Steps

Now that you have generated data:

1. **Explore the data** - Open `01_CausalTrace_Quickstart.ipynb`
2. **Run evaluation** - Open `02_Full_Evaluation.ipynb` 
3. **Compare baselines** - Open `03_Baseline_Comparison.ipynb`
4. **Train detectors** - Run `experiments/train_gnn.py`

## Troubleshooting

### "Docker not found"

Install Docker Desktop:
- macOS: https://docs.docker.com/desktop/install/mac-install/
- Linux: https://docs.docker.com/desktop/install/linux-install/
- Windows: https://docs.docker.com/desktop/install/windows-install/

### "API rate limit exceeded"

Use simulated writes:
```bash
export SIMULATE_WRITES=true
python generate.py --num-trajectories 100
```

### "Stripe production key detected"

Make sure to use test mode keys only:
```bash
# Wrong: sk_live_...
# Correct: sk_test_...
export STRIPE_API_KEY=sk_test_your_key_here
```

### More Help

See full documentation:
- `../DATA_GENERATION.md` - Complete guide
- `../causalbench-generator/README.md` - Generator details
- `../causalbench-generator/experiments/` - Example scripts